### Training a Random Forest Regressor model
Training a random forest regressor that predicts FPL points for the upcoming GW for players. This is a general model, and uses position as one of the predictors. A future step might be to produce a separate model for each position so that position-specific features can be better considered.

Rolling game statistics are key to the model - they will be computed on the previous three games for each player, and used as predictor features.

In [42]:
import os
import torch
import pandas as pd
import numpy as np
from model import AdvancedLSTM
import pickle

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [4]:
X_train = torch.load(data_path + '/X_train.pt', weights_only=True)
y_train = torch.load(data_path + '/y_train.pt', weights_only=True)
train_mapping = pd.read_csv(data_path + '/train_mapping.csv')

X_val = torch.load(data_path + '/X_val.pt', weights_only=True)
y_val = torch.load(data_path + '/y_val.pt', weights_only=True)

X_test = torch.load(data_path + '/X_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')

In [5]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: torch.Size([169711, 5, 56])
y_train shape: torch.Size([169711, 1])
X_val shape: torch.Size([11384, 5, 56])
y_val shape: torch.Size([11384, 1])


In [101]:
# Find all test_mapping rows for Mohamed Salah
salah_mapping = test_mapping[test_mapping['name'] == 'alexander_isak']

# Get the sequence indices for Salah
salah_sequence_indices = salah_mapping['sequence_idx'].values

# Extract the corresponding X_test data for Salah
salah_x_test = X_test[salah_sequence_indices]

# You can also get the corresponding y_test data
salah_y_test = y_test[salah_sequence_indices]

In [102]:
last_1_assists_idx = 2
last_1_goals_scored_idx = 8

In [103]:
#sanity check to see there is no data leakage
for i in range(salah_x_test.shape[0]):
    last_1_assists = salah_x_test[i, -1, last_1_assists_idx]
    last_1_goals_scored = salah_x_test[i, -1,  last_1_goals_scored_idx]
    
    print(f"Gameweek {i + 1}: Last 1 Assists: {last_1_assists}, Last 1 Goals Scored: {last_1_goals_scored}, Total Points: {salah_y_test[i]}")

Gameweek 1: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([5.])
Gameweek 2: Last 1 Assists: 0.25, Last 1 Goals Scored: 0.0, Total Points: tensor([2.])
Gameweek 3: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([9.])
Gameweek 4: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([1.])
Gameweek 5: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([2.])
Gameweek 6: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([2.])
Gameweek 7: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([6.])
Gameweek 8: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([9.])
Gameweek 9: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.25, Total Points: tensor([11.])
Gameweek 10: Last 1 Assists: 0.25, Last 1 Goals Scored: 0.25, Total Points: tensor([2.])
Gameweek 11: Last 1 Assists: 0.0, Last 1 Goals Scored: 0.0, Total Points: tensor([1.])
Gameweek 12: Last 1 Assists: 0.0, Last 1 Goal

In [104]:
best_model_data = torch.load("best_model.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_3764/2890380057.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model.pth",

In [105]:
print(best_model_data['hidden_dim'])
print(best_model_data['num_layers'])
print(best_model_data['num_fc_layers'])

192
1
3


In [106]:
model = AdvancedLSTM(
    hidden_dim=best_model_data['hidden_dim'],
    num_layers=best_model_data['num_layers'],
    input_dim= best_model_data['input_dim'],
    output_dim=1,
    num_fc_layers=best_model_data['num_fc_layers'],
) 
model.load_state_dict(best_model_data['model_state_dict'])

/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


<All keys matched successfully>

In [107]:
model.eval()

AdvancedLSTM(
  (lstm): LSTM(56, 192, batch_first=True, dropout=0.3)
  (fc_layers): ModuleList(
    (0): Linear(in_features=192, out_features=96, bias=True)
    (1): Linear(in_features=96, out_features=48, bias=True)
    (2): Linear(in_features=48, out_features=1, bias=True)
  )
  (batch_norms): ModuleList(
    (0): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (relu): ReLU()
)

In [108]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

torch.Size([169711, 5, 56])
torch.Size([11384, 5, 56])
torch.Size([11567, 5, 56])


In [109]:
test = pd.read_csv(data_path + '/test_data.csv')

In [110]:
predictions = model(X_test).detach().numpy()
print(predictions.shape)
print(y_test.shape)

(11567, 1)
torch.Size([11567, 1])


In [111]:
print(torch.mean(y_test))
print(torch.var(y_test))    

tensor(2.7053)
tensor(8.3782)


In [112]:
print(np.mean(predictions))
print(np.var(predictions))

2.3046472
3.2856576


In [113]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(y_test, predictions)
print(f"RMSE: {rmse}")

mae = mean_absolute_error(y_test, predictions)
print(f"MAE: {mae}")


RMSE: 2.601583957672119
MAE: 1.629275918006897


In [114]:
X_test.shape

torch.Size([11567, 5, 56])

In [115]:
salah_y_test[5]

tensor([2.])

In [116]:
salah_x_test[5]

tensor([[4.0000e-01, 1.0000e+00, 2.5000e-01, 0.0000e+00, 3.0719e-01, 1.0000e+00,
         6.3195e-02, 0.0000e+00, 0.0000e+00, 1.1732e-01, 1.5648e-01, 1.0000e+00,
         0.0000e+00, 1.0000e+00, 3.0151e-02, 3.3333e-01, 0.0000e+00, 1.4286e-01,
         0.0000e+00, 1.8677e-01, 3.3333e-01, 3.7474e-02, 0.0000e+00, 0.0000e+00,
         6.0870e-02, 8.7491e-02, 3.3333e-01, 0.0000e+00, 3.3333e-01, 1.7910e-02,
         2.0690e-01, 0.0000e+00, 1.2500e-01, 0.0000e+00, 1.4151e-01, 2.0000e-01,
         0.0000e+00, 0.0000e+00, 5.8662e-02, 0.0000e+00, 2.0000e-01, 1.0843e-01,
         0.0000e+00, 4.7619e-02, 0.0000e+00, 3.8256e-02, 4.7619e-02, 5.9927e-03,
         0.0000e+00, 0.0000e+00, 9.0361e-03, 0.0000e+00, 3.0769e-02, 2.5907e-03,
         0.0000e+00, 3.0000e+00],
        [6.0000e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 2.1569e-01, 0.0000e+00,
         6.7876e-02, 1.1111e-01, 0.0000e+00, 1.8715e-01, 4.0342e-02, 1.0000e+00,
         0.0000e+00, 5.0000e-01, 2.4623e-01, 2.5000e-01, 0.0000e+00, 1.4286

In [117]:
for i in range(0, len(salah_x_test)):
    input_to_game = salah_x_test[i].unsqueeze(0)
    print(f"Gameweek {i + 1}:")
    print("prediction", model(input_to_game).detach().numpy())
    print("actual", salah_y_test[i].item())

Gameweek 1:
prediction [[1.3431102]]
actual 5.0
Gameweek 2:
prediction [[2.7924676]]
actual 2.0
Gameweek 3:
prediction [[1.344689]]
actual 9.0
Gameweek 4:
prediction [[1.525836]]
actual 1.0
Gameweek 5:
prediction [[1.3104534]]
actual 2.0
Gameweek 6:
prediction [[1.3288589]]
actual 2.0
Gameweek 7:
prediction [[10.850378]]
actual 6.0
Gameweek 8:
prediction [[3.0352974]]
actual 9.0
Gameweek 9:
prediction [[4.876954]]
actual 11.0
Gameweek 10:
prediction [[1.5729234]]
actual 2.0
Gameweek 11:
prediction [[1.5223283]]
actual 1.0
Gameweek 12:
prediction [[8.011631]]
actual 11.0
Gameweek 13:
prediction [[11.233089]]
actual 8.0
Gameweek 14:
prediction [[7.8003016]]
actual 11.0
Gameweek 15:
prediction [[14.57238]]
actual 17.0
Gameweek 16:
prediction [[8.634826]]
actual 8.0
Gameweek 17:
prediction [[7.811783]]
actual 7.0
Gameweek 18:
prediction [[8.932868]]
actual 7.0
Gameweek 19:
prediction [[15.954845]]
actual 16.0
Gameweek 20:
prediction [[1.3473856]]
actual 2.0
Gameweek 21:
prediction [[8.7069

In [95]:
import pulp
import pandas as pd
import numpy as np
import os
import torch

data_path = os.getcwd() + '/processed_data'

def make_available_players_df(this_season_player_df, last_season_player_df):
    
    last_season_player_df = last_season_player_df[last_season_player_df.minutes > 0]
    last_season_player_df = last_season_player_df[['name', "total_points"]]
    last_season_player_df.rename(columns={'total_points': "total_points_last_season"},
                                inplace=True)
    
    available_players_df = pd.merge(this_season_player_df,
                                    last_season_player_df,
                                   on='name', how='left')

    # First attempt: fill by position and value groups
    available_players_df['total_points_last_season'] = available_players_df.groupby(['position_encoded', 'value'])['total_points_last_season'].transform(lambda x: x.fillna(x.mean()))
    
    # Second attempt: fill by position only if still NaN
    available_players_df['total_points_last_season'] = available_players_df.groupby(['position_encoded'])['total_points_last_season'].transform(lambda x: x.fillna(x.mean()))
    
    nan_values = available_players_df[available_players_df['total_points_last_season'].isna()]
    print("Players with NaN total_points_last_season:", nan_values['name'].unique())
    print("Number of NaN values remaining:", len(nan_values))
    
    return available_players_df

def get_cheapest_players(player_df):
    
    cheapest_player_names = []
    total_cost = 0
    
    # for each position, sort the players by cost (in ascending order)
    # then, get the player with the most number of points
    
    for position, group in player_df.groupby('position_encoded'):
        cheapest_players =  group[(group.value == group.value.min())]
        top_cheapest_player = cheapest_players[cheapest_players['total_points'] == cheapest_players['total_points'].max()]

        cheapest_player_name = top_cheapest_player['name'].values[0]
        
        cheapest_player_names += [cheapest_player_name]
        total_cost += top_cheapest_player.value.values[0]
        
        print(position, ": ", cheapest_player_name )
        
    return cheapest_player_names, total_cost


def make_decision_variables(player_df):
    return [pulp.LpVariable(i, cat="Binary") for i in player_df['name']]


def make_optimization_function(player_df, decision_variables):
    op_func = ""

    for i, player in enumerate(decision_variables):
        op_func += player_df.total_points_last_season[i] * player

    return op_func


def make_cash_constraint(player_df, decision_variables, available_cash):
    total_paid = ""
    for rownum, row in player_df.iterrows():
        for i, player in enumerate(decision_variables):
            if rownum == i:
                formula = row['value']*player
                total_paid += formula

    return (total_paid <= available_cash)


def make_player_constraint(position, n, decision_variables, player_df):
    
    total_n = ""
    
    player_positions = player_df.position_encoded
    
    for i, player in enumerate(decision_variables):
        if player_positions[i] == position:
            total_n += 1*player
            
    return(total_n == n)


def add_team_constraint(prob, player_df, decision_variables):

    for team, group in player_df.groupby('team_x'):
        team_total = ''
        
        for player in decision_variables:
            if player.name in group['name'].values:
                formula = 1*player
                team_total += formula
                
        
        prob += (team_total <= 3)


def solve_optimization_problem(available_players_df, bench_cost):

    available_cash = 1000 - bench_cost

    prob = pulp.LpProblem('InitialTeam', pulp.LpMaximize)
    print("Available cash:", available_cash)
    decision_variables = make_decision_variables(available_players_df)
    print("Decision variables:", decision_variables)
    prob += make_optimization_function(available_players_df, decision_variables)
    print("Optimization function:", prob.objective)
    prob += make_cash_constraint(available_players_df, decision_variables, available_cash)
    prob += make_player_constraint(0, 1, decision_variables, available_players_df) 
    prob += make_player_constraint(1, 4, decision_variables, available_players_df) 
    prob += make_player_constraint(2, 4, decision_variables, available_players_df) 
    prob += make_player_constraint(3, 2, decision_variables, available_players_df)

    add_team_constraint(prob, available_players_df, decision_variables)

    prob.writeLP('InitialTeam.lp')
    prob.solve()

    return prob


def get_initial_team(prob, player_df):
    variable_names = [v.name for v in prob.variables()]
    variable_values = [v.varValue for v in prob.variables()]

    # Create the decision variables DataFrame
    decision_df = pd.DataFrame({"name": variable_names, "selected": variable_values})

    # Perform merge
    initial_team = pd.merge(decision_df, player_df, on="name", how='left')
    initial_team = initial_team[initial_team["selected"] == 1.0]

    return initial_team


def make_predicted_table(y_test, y_pred, gw_df):
    '''
    Create a DataFrame for LSTM model predictions.
    This needs to keep track of the Gameweek (GW) and player names.
    '''
    test_mapping = pd.read_csv(data_path + '/test_mapping.csv')
    predictions_df = test_mapping.copy()
    predictions_df['actual'] = y_test
    predictions_df['predicted'] = y_pred 
    #predictions_df = predictions_df[predictions_df['minutes'] > 0]  # filter out players who did not play
    predictions_df.rename(columns={'prediction_gw': 'GW'}, inplace=True)
    gameweek_1 = gw_df[gw_df['GW'] == 1][['name', 'total_points', 'value', 'GW', 'element', 'season_x', 'team_x', 'minutes', 'position_encoded', 'last_1_goals_scored']].copy()
    gameweek_1 = gameweek_1[gameweek_1['minutes'] > 0]  # filter out players who did not play

    # Add the missing columns to match predictions_df structure
    gameweek_1['actual'] = gameweek_1['total_points']
    gameweek_1['predicted'] = np.nan
    gameweek_1['sequence_idx'] = np.nan  # No sequence for GW1
    gameweek_1['padding_used'] = np.nan  # No padding info for GW1

    # Reorder columns to match predictions_df
    gameweek_1 = gameweek_1[['sequence_idx', 'element', 'season_x', 'name', 'GW', 'team_x', 'value', 'minutes', 'padding_used', 'actual', 'predicted', 'last_1_goals_scored', 'position_encoded']]

    # Combine with existing predictions_df
    predictions_df_complete = pd.concat([gameweek_1, predictions_df], ignore_index=True)

    # Sort by player and gameweek for better organization
    predictions_df_complete = predictions_df_complete.sort_values(['name', 'GW']).reset_index(drop=True)

    # Update the predictions_df reference
    predictions_df = predictions_df_complete
    predictions_df = predictions_df.drop_duplicates(subset=['name', 'GW'], keep='last')
    
    return predictions_df


def get_suggested_transfer(predicted_df, team_list, current_money):
    
    predicted_diff = 0
    money_change = 0
    suggested_in = ''
    suggested_out = ''
    team_df = predicted_df[(predicted_df['name'].isin(team_list))]


    teams_dict = {}
    for i, row in team_df.iterrows():
        if row.team_x not in teams_dict:
            teams_dict[row.team_x] = [row['name']]
        else:
            teams_dict[row.team_x].append(row['name'])


    for position in [1, 2, 3]:
        
        # don't bother about keepers, variance in scores is not that great
        # so, save the free transfer for other positions

        player_df = predicted_df[predicted_df.position_encoded==position].sort_values('predicted', ascending=False).reset_index()
        lowest_pos = 0
        player_names = team_df[team_df.position_encoded==position]['name'].values
        
        # loop through the players for this position, and get the rank (row number) of the player with the lowest predicted score
        for p in player_names:
            player_pos = player_df[player_df['name']==p].index[0]
            if player_pos > lowest_pos:
                lowest_pos = player_pos
                potential_out = p
                potential_out_cost = team_df[team_df['name']==p].value.values[0]
                potential_out_team = team_df[team_df['name']==p].team_x.values[0]

            elif len(player_names) <= 1:
                potential_out_cost = 0
                potential_out_team = 'none'
                potential_out = 'none'
                
        # get all players above this player
        potential_players = player_df[:lowest_pos]
        
        # only keep players that we can afford
        potential_players = potential_players[potential_players.value <= potential_out_cost + current_money]
        
        # only keep players who played (need a better way of doing this)
        potential_players = potential_players[potential_players.minutes > 0]

        # get the prediction difference for each suggested player
        # select the one with the highest difference as the suggested transfer (compare across positons)
        
        potential_out_predicted = team_df[team_df['name']==p].predicted.values[0]

        for i, row in potential_players.iterrows():
            # skip if it is a player we already have
            if row['name'] in team_list:
                continue



            # if there are no other players of the same team, it's ok to consider this player
            # if not, check whether there are 3 players of the same team already
            if row.team_x not in teams_dict:
                pass
            else:
                if len(teams_dict[row.team_x]) == 3:
                    # if there are already 3 players of the same team,
                    # can't take another player of the same team
                    # unless the suggested_out is the same team as suggested_in (direct swap)
                   
                    if row.team_x == potential_out_team:
                        pass
                    else:
                        continue
                else:
                    pass
                
            
            # check for difference in predictions
            if row.predicted - potential_out_predicted > predicted_diff:
                predicted_diff = row.predicted - potential_out_predicted
                suggested_in = row['name']
                suggested_out = potential_out
                
                # calculate change in money
                money_change = potential_out_cost - row.value
                
    return suggested_in, suggested_out, money_change


def get_score(team_list, gw_df, sort_by='predicted'):
    
    gw_score = gw_df[gw_df['name'].isin(team_list)].actual.sum() \
        + gw_df[(gw_df['name'].isin(team_list)) & (gw_df['position_encoded']!= 0)].sort_values(sort_by, ascending=False).head(1).actual.values[0]

    print(gw_df[gw_df['name'].isin(team_list)][['name', 'actual', 'predicted']])
    print("total_score for gameweek", gw_df['GW'].values[0], ":", gw_score)
    return gw_score



def get_performance(team_list, starting_money, gw_list,
                   prediction_df):
    
    current_money = starting_money
    total_score = 0
    
    
    in_list = []
    out_list = []
    score_list = []
    unplayed_list = []
    
    
    for gw in gw_list:
        print("Gameweek:", gw)
        print("my team:", team_list)
        gw_df = prediction_df[prediction_df.GW==gw]
        money_change = 0
        suggested_in = ''
        suggested_out = ''
        if gw > 1:
            

            suggested_in, suggested_out, money_change = get_suggested_transfer(gw_df, team_list, current_money)
        
            current_money += money_change

            team_list.append(suggested_in)
            team_list.remove(suggested_out)
            
        

        ## Calculate scores
        
        gw_score = get_score(team_list, gw_df)

        print("suggested in:", suggested_in)
        print("suggested out:", suggested_out)
        out_list.append(suggested_out)
        in_list.append(suggested_in)
        score_list.append(gw_score)
        
        total_score += gw_score
        
    out_df = pd.DataFrame({'GW': gw_list,
                          'player_in': in_list,
                          'player_out': out_list,
                          'total_score': score_list})

    
    return out_df, total_score



def get_season_performance(y_test, predictions, remaining_lagged_features):
    """
    Get the season performance of the team based on the predictions.
    """ 
    previous_season = pd.read_csv(data_path + '/val_data.csv')
    test = pd.read_csv(data_path + '/test_data.csv')

    group_key = 'name'
    first_cols = ['team_x','season_x', 'position_encoded', 'element', 'value', 'position', 'was_home']

    # Only use non-grouping columns in aggregation
    agg_dict = {
        col: 'first' if col in first_cols else 'sum'
        for col in test.columns
        if col != group_key
    }

    summed_test = test.groupby('name', as_index=False).agg(agg_dict)
    summed_last_season = previous_season.groupby('name', as_index=False).agg(agg_dict)
    summed_last_season['value'] = summed_last_season['value'].astype(int)
    summed_test['value'] = summed_test['value'].astype(int)

    available_players_df = make_available_players_df(summed_test, summed_last_season)

    bench_player_names, bench_cost = get_cheapest_players(available_players_df)
    print("Bench players:", bench_player_names)
    print("Bench cost:", bench_cost)

    predicted_df = make_predicted_table(y_test, predictions, test[remaining_lagged_features + ['name', 'GW', 'team_x', 'total_points', 'value', 'minutes', 'season_x', 'element']])
    print("length available players df", len(available_players_df))
    prob = solve_optimization_problem(available_players_df, bench_cost)
    initial_team_df = get_initial_team(prob, available_players_df)
    my_team = initial_team_df['name'].tolist()

    print("My team:", my_team)
    gameweeks = (test.GW).unique()
    print("Gameweeks:", len(gameweeks))
    starting_money = 1000 - bench_cost - initial_team_df.value.sum()
    print("Starting money:", starting_money)

    xgb_cv_perf, total_score = get_performance(my_team, starting_money, gameweeks,
                    predicted_df)
    
    return xgb_cv_perf, total_score

def season_performance_with_unlimited_transfers(y_test, predictions, remaining_lagged_features):
    """
    Get the season performance of the team based on the predictions with unlimited transfers.
    Creates a completely new optimal team for each gameweek using that gameweek's predictions.
    """
    previous_season = pd.read_csv(data_path + '/val_data.csv')
    test = pd.read_csv(data_path + '/test_data.csv')

    group_key = 'name'
    first_cols = ['team_x','season_x', 'position_encoded', 'element', 'value', 'position', 'was_home']

    # Only use non-grouping columns in aggregation
    agg_dict = {
        col: 'first' if col in first_cols else 'sum'
        for col in test.columns
        if col != group_key
    }

    summed_test = test.groupby('name', as_index=False).agg(agg_dict)
    summed_last_season = previous_season.groupby('name', as_index=False).agg(agg_dict)
    summed_last_season['value'] = summed_last_season['value'].astype(int)
    summed_test['value'] = summed_test['value'].astype(int)

    # Create the predicted table for all gameweeks
    predicted_df = make_predicted_table(y_test, predictions, test[remaining_lagged_features + ['name', 'GW', 'team_x', 'total_points', 'value', 'minutes', 'season_x', 'element']])
    
    # Get available players (merge current season with previous season data)
    available_players_df = make_available_players_df(summed_test, summed_last_season)
    
    # Get bench players and cost (needed for budget calculation)
    bench_player_names, bench_cost = get_cheapest_players(available_players_df)
    print("Bench players:", bench_player_names)
    print("Bench cost:", bench_cost)
    
    # Available budget for main team (1000 - bench cost)
    available_budget = 1000 - bench_cost
    
    gameweeks = sorted(test['GW'].unique())
    total_score = 0
    gw_scores = []
    gw_teams = []
    
    print(f"Simulating season with unlimited transfers for {len(gameweeks)} gameweeks")
    print(f"Available budget per gameweek: {available_budget}")
    
    for gw in gameweeks:
        print(f"\n--- Gameweek {gw} ---")
        
        # Get predictions for this specific gameweek
        gw_predictions = predicted_df[predicted_df['GW'] == gw].copy()

        # Create a temporary available players dataframe with this gameweek's predictions
        # Merge current gameweek predictions with player info
        gw_player_data = pd.merge(
            available_players_df[['name', 'team_x', 'position_encoded', 'value', 'total_points_last_season']],
            gw_predictions[['name', 'predicted']],
            on='name',
            how='inner'
        )
        
        # Only consider players who are predicted to play (have predictions)
        gw_player_data = gw_player_data.dropna(subset=['predicted'])
        
        print(f"Players available for GW {gw}: {len(gw_player_data)}")
        
        if len(gw_player_data) == 0:
            prob = solve_optimization_problem(available_players_df, bench_cost)
            initial_team_df = get_initial_team(prob, available_players_df)
            my_team = initial_team_df['name'].tolist()
            print("My team:", my_team)
            gw_score = get_score(my_team, gw_predictions, sort_by='actual')
            total_score += gw_score
            gw_teams.append(my_team)
            gw_scores.append(gw_score)
            continue
        
        # Solve optimization problem for this gameweek using predicted points
        try:
            prob = solve_optimization_problem_for_gameweek(gw_player_data, bench_cost, gw)
            
            if prob.status == 1:  # Optimal solution found
                # Get the optimal team for this gameweek
                optimal_team_df = get_initial_team(prob, gw_player_data)
                optimal_team_names = optimal_team_df['name'].tolist()
                
                # Calculate score for this gameweek using actual points
                gw_actual_data = gw_predictions[gw_predictions['name'].isin(optimal_team_names)]
                gw_score = gw_actual_data['actual'].sum()
                print(gw_actual_data[['name', 'actual', 'predicted']])
                # Add captain bonus (best performing outfield player gets double points)
                outfield_players = gw_actual_data[gw_actual_data['position_encoded'] != 0]
                if len(outfield_players) > 0:
                    captain_bonus = outfield_players['actual'].max()
                    gw_score += captain_bonus
                
                total_score += gw_score
                gw_scores.append(gw_score)
                gw_teams.append(optimal_team_names)
                
                print(f"GW {gw} optimal team: {optimal_team_names}")
                print(f"GW {gw} score: {gw_score}")
                print(f"Team cost: {optimal_team_df['value'].sum()}")
                
            else:
                print(f"Could not find optimal solution for GW {gw}")
                gw_scores.append(0)
                gw_teams.append([])
                
        except Exception as e:
            print(f"Error solving optimization for GW {gw}: {e}")
            gw_scores.append(0)
            gw_teams.append([])
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'team': gw_teams,
        'gw_score': gw_scores
    })
    
    print(f"\n=== UNLIMITED TRANSFERS SEASON SUMMARY ===")
    print(f"Total season score: {total_score}")
    print(f"Average GW score: {np.mean(gw_scores):.2f}")
    print(f"Best GW score: {max(gw_scores) if gw_scores else 0}")
    print(f"Worst GW score: {min(gw_scores) if gw_scores else 0}")
    
    return results_df, total_score


def solve_optimization_problem_for_gameweek(gw_player_data, bench_cost, gw_num):
    """
    Solve optimization problem for a specific gameweek using predicted points as the objective.
    """
    available_cash = 1000 - bench_cost
    
    prob = pulp.LpProblem(f'OptimalTeam_GW{gw_num}', pulp.LpMaximize)
    
    # Create decision variables
    decision_variables = [pulp.LpVariable(name, cat="Binary") for name in gw_player_data['name']]
    
    # Objective function: maximize predicted points for this gameweek
    objective = ""
    for i, player in enumerate(decision_variables):
        objective += gw_player_data.iloc[i]['predicted'] * player
    prob += objective
    
    # Budget constraint
    budget_constraint = ""
    for i, player in enumerate(decision_variables):
        budget_constraint += gw_player_data.iloc[i]['value'] * player
    prob += (budget_constraint <= available_cash)
    
    # Position constraints (1 GK, 4 DEF, 4 MID, 2 FWD)
    for position in [0, 1, 2, 3]:
        position_constraint = ""
        required_count = [1, 4, 4, 2][position]  # GK, DEF, MID, FWD
        
        for i, player in enumerate(decision_variables):
            if gw_player_data.iloc[i]['position_encoded'] == position:
                position_constraint += 1 * player
        
        prob += (position_constraint == required_count)
    
    # Team constraint (max 3 players per team)
    for team in gw_player_data['team_x'].unique():
        team_constraint = ""
        team_players = gw_player_data[gw_player_data['team_x'] == team]
        
        for i, player in enumerate(decision_variables):
            if gw_player_data.iloc[i]['name'] in team_players['name'].values:
                team_constraint += 1 * player
        
        prob += (team_constraint <= 3)
    
    # Solve the problem
    prob.solve()  # Silent solver
    
    return prob


In [96]:
remaining_lagged_features = pickle.load(open(data_path + '/remaining_lagged_features.pkl', 'rb'))
remaining_lagged_features

['last_1_assists',
 'last_3_assists',
 'last_5_assists',
 'last_all_assists',
 'last_1_bonus',
 'last_3_bonus',
 'last_5_bonus',
 'last_all_bonus',
 'last_1_bps',
 'last_3_bps',
 'last_5_bps',
 'last_all_bps',
 'last_1_creativity',
 'last_3_creativity',
 'last_all_creativity',
 'last_1_clean_sheets',
 'last_3_clean_sheets',
 'last_5_clean_sheets',
 'last_all_clean_sheets',
 'last_1_goals_conceded',
 'last_3_goals_conceded',
 'last_5_goals_conceded',
 'last_all_goals_conceded',
 'last_1_goals_scored',
 'last_3_goals_scored',
 'last_5_goals_scored',
 'last_all_goals_scored',
 'last_1_ict_index',
 'last_3_ict_index',
 'last_all_ict_index',
 'last_1_influence',
 'last_3_influence',
 'last_5_influence',
 'last_1_minutes',
 'last_3_minutes',
 'last_1_threat',
 'last_3_threat',
 'last_all_threat',
 'last_1_red_cards',
 'last_3_red_cards',
 'last_5_red_cards',
 'last_all_red_cards',
 'last_1_yellow_cards',
 'last_3_yellow_cards',
 'last_5_yellow_cards',
 'last_all_yellow_cards',
 'last_1_resul

In [97]:
scores, total_score = get_season_performance(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=remaining_lagged_features
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
0.0 :  daniel_bentley
1.0 :  luke_thomas
2.0 :  matheus_franca_de_oliveira
3.0 :  daniel_jebbison
Bench players: ['daniel_bentley', 'luke_thomas', 'matheus_franca_de_oliveira', 'daniel_jebbison']
Bench cost: 167
length available players df 562
Available cash: 833
Decision variables: [aaron_cresswell, aaron_ramsdale, aaron_wan_bissaka, abdoulaye_doucoure, abdukodir_khusanov, abdul_fatawu, adam_armstrong, adam_lallana, adam_smith, adam_webster, adam_wharton, adama_traore, albert_gronbaek, alejandro_garnacho, alex_iwobi, alex_mccarthy, alex_moreno_lopera, alex_palmer, alex_scott, alexander_isak, alexis_mac_allister, alfie_dorrington, alfie_pond, ali_al_hamadi, alisson_ramses_becker, alphonse_areola, altay_bayindir, amad_diallo, amadou_onana, andre_onana, andre_trindade_da_costa_neto, andreas_hoelgebaum_pereira, andres_garcia, andrew_robertson, andy_irving, anthony_elanga, anthony_gordon, antoine_semenyo, anton

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_3764/3251774381.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions_df_complete = pd.concat([gameweek_1, predictions_df], ignore_index=True)


                  name  actual  predicted
1173    benjamin_white     1.0   4.869621
1558       bukayo_saka     4.0   1.937658
2103       cole_palmer     8.0   1.540170
2496     danny_welbeck     6.0   5.462485
3047     dwight_mcneil     2.0  10.737935
5535  joachim_andersen     2.0   1.310453
5895   jordan_pickford     2.0   3.955773
6457      kevin_schade     1.0   1.400104
7365     manuel_akanji     1.0   4.349134
9188     ollie_watkins     6.0   2.309385
total_score for gameweek 5.0 : 35.0
suggested in: dwight_mcneil
suggested out: phil_foden
Gameweek: 6.0
my team: ['benjamin_white', 'bukayo_saka', 'cole_palmer', 'jarrad_branthwaite', 'joachim_andersen', 'jordan_pickford', 'ollie_watkins', 'danny_welbeck', 'manuel_akanji', 'kevin_schade', 'dwight_mcneil']
                  name  actual  predicted
1559       bukayo_saka     3.0   1.503191
2104       cole_palmer    25.0  23.133987
2497     danny_welbeck     2.0   1.839026
3048     dwight_mcneil    15.0   8.505720
4916      james_justi

In [98]:
scores

,GW,player_in,player_out,total_score
0,1.0,,,33.0
1,2.0,danny_welbeck,jean_philippe_mateta,72.0
2,3.0,manuel_akanji,william_saliba,28.0
3,4.0,kevin_schade,son_heung_min,39.0
4,5.0,dwight_mcneil,phil_foden,35.0
5,6.0,james_justin,jarrad_branthwaite,106.0
6,7.0,jarrod_bowen,kevin_schade,90.0
7,8.0,michael_keane,joachim_andersen,57.0
8,9.0,alexander_isak,ollie_watkins,62.0
9,10.0,dominic_solanke_mitchell,danny_welbeck,55.0


In [100]:
total_score.item()

1943.0

In [56]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=remaining_lagged_features
)

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_3764/1970781112.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions_df_complete = pd.concat([gameweek_1, predictions_df], ignore_index=True)


Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
0.0 :  daniel_bentley
1.0 :  luke_thomas
2.0 :  matheus_franca_de_oliveira
3.0 :  daniel_jebbison
Bench players: ['daniel_bentley', 'luke_thomas', 'matheus_franca_de_oliveira', 'daniel_jebbison']
Bench cost: 167
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 833

--- Gameweek 1.0 ---
Players available for GW 1.0: 316
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/d2a720be194f4d7593f6ac878e257a89-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/d2a720be194f4d7593f6ac878e257a89-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 1927 RHS
At line 1953 BOUNDS
At

In [57]:
predicted_df = make_predicted_table(y_test, predictions, test[remaining_lagged_features + ['name', 'GW', 'team_x', 'total_points', 'value', 'minutes', 'season_x', 'element']])

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_3764/1970781112.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions_df_complete = pd.concat([gameweek_1, predictions_df], ignore_index=True)


In [58]:
total_score.item()

3543.0

In [59]:
scores

,team,gw_score
0,"[cody_gakpo, craig_dawson, guglielmo_vicario, ...",28.0
1,"[danny_welbeck, dominik_szoboszlai, ederson_sa...",48.0
2,"[adama_traore, bernardo_veiga_de_carvalho_e_si...",58.0
3,"[antoine_semenyo, ilkay_gundogan, kai_havertz,...",50.0
4,"[aaron_ramsdale, amad_diallo, danny_welbeck, d...",46.0
5,"[antonee_robinson, bernd_leno, brennan_johnson...",142.0
6,"[bukayo_saka, emiliano_martinez_romero, facund...",121.0
7,"[alejandro_garnacho, bart_verbruggen, danny_we...",107.0
8,"[alex_iwobi, alexander_isak, bernd_leno, carlo...",95.0
9,"[aaron_ramsdale, antoine_semenyo, brennan_john...",120.0
